# Milestone 021 on Colab

Use this notebook after switching the Colab runtime to `TPU`. The notebook clones the repo, installs TPU JAX, stages the shard dataset from Drive into `/content`, and runs `experiments/021_tpu_fineweb_edu_multi_shard.py`.


In [ ]:
!git clone https://github.com/marcoshernanz/llm-lab.git /content/llm-lab
%cd /content/llm-lab
!git checkout milestone-021
!python -m pip install -q -U pip
!python -m pip install -q -U "jax[tpu]" flax optax numpy pyarrow datasets huggingface-hub equinox

In [ ]:
"""Check that Colab attached a TPU-backed JAX runtime."""

import jax

print("default_backend=", jax.default_backend())
print("devices=")
for device in jax.devices():
    print("  ", device)


In [ ]:
"""Copy the tokenizer and token shards from Drive into fast local storage."""

from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

# Edit these two paths to match where you keep the milestone-020 artifacts in Drive.
DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384")
DRIVE_TOKENIZER_PATH = Path("/content/drive/MyDrive/llm-lab/artifacts/tokenizers/fineweb_edu_sample10bt_bpe_16384.json")

LOCAL_DATA_ROOT = Path("/content/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384")
LOCAL_TOKENIZER_PATH = Path("/content/llm-lab/artifacts/tokenizers/fineweb_edu_sample10bt_bpe_16384.json")

if not DRIVE_DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_ROOT}")
if not DRIVE_TOKENIZER_PATH.exists():
    raise FileNotFoundError(f"Tokenizer not found at {DRIVE_TOKENIZER_PATH}")

if LOCAL_DATA_ROOT.exists():
    shutil.rmtree(LOCAL_DATA_ROOT)
LOCAL_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_DATASET_ROOT, LOCAL_DATA_ROOT)

LOCAL_TOKENIZER_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_TOKENIZER_PATH, LOCAL_TOKENIZER_PATH)

print("local_dataset_root=", LOCAL_DATA_ROOT)
print("local_tokenizer_path=", LOCAL_TOKENIZER_PATH)


In [ ]:
!python experiments/021_tpu_fineweb_edu_multi_shard.py \
  --token-shard-root /content/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384 \
  --tokenizer-path /content/llm-lab/artifacts/tokenizers/fineweb_edu_sample10bt_bpe_16384.json \
  --max-train-shards 10 \
  --train-steps 2000 \
  --train-chunk-length 40

In [ ]:
"""Display the newest loss curve written by the milestone-021 run."""

from IPython.display import SVG, display
from pathlib import Path

run_dirs = sorted(Path("/content/llm-lab/artifacts/experiments/021_tpu_fineweb_edu_multi_shard").glob("*"))
if not run_dirs:
    raise FileNotFoundError("No milestone-021 artifact directory was created.")

latest_run_dir = run_dirs[-1]
print("latest_run_dir=", latest_run_dir)
display(SVG(filename=str(latest_run_dir / "loss_curve.svg")))
print((latest_run_dir / "sample.txt").read_text(encoding="utf-8"))
